In [ ]:
# Install the allenai fork of lerobot that contains MolmoAct2 policy support.
!pip install -q 'git+https://github.com/allenai/lerobot.git@molmoact2-policy#egg=lerobot[molmoact2]'

# PyAV >= 12.0 required for av.option.Option used in lerobot's pyav_utils.py
!pip install -q 'av>=12.0.0' --upgrade

DEPRECATION: git+https://github.com/allenai/lerobot.git@molmoact2-policy#egg=lerobot[molmoact2] contains an egg fragment with a non-PEP 508 name pip 25.0 will enforce this behaviour change. A possible replacement is to use the req @ url syntax, and remove the egg fragment. Discussion can be found at https://github.com/pypa/pip/issues/11617
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Patch pyav_utils.py so that the `av.option.Option` return-type annotation
# is not evaluated at import time (same trick as Pi0.5's vision-tower fix).
# This is needed because lerobot may pin av to a version that lacks av.option.
import glob

for path in glob.glob("/usr/local/lib/python3.*/dist-packages/lerobot/datasets/pyav_utils.py"):
    src = open(path).read()
    if "from __future__ import annotations" not in src:
        open(path, "w").write("from __future__ import annotations\n" + src)
        print(f"Patched: {path}")
    else:
        print(f"Already patched: {path}")

Already patched: /usr/local/lib/python3.13/dist-packages/lerobot/datasets/pyav_utils.py


In [ ]:
from google.colab import userdata
from huggingface_hub import login
login(token=userdata.get('HF_TOKEN'))
print("Logged in to HuggingFace.")

Logged in to HuggingFace.


In [ ]:
# ── Dataset ───────────────────────────────────────────────────────────────────
DATASET_REPO_ID = "subhodipsaha/so101_pick_place_07_16"

# ── Pretrained checkpoint ─────────────────────────────────────────────────────
CHECKPOINT_PATH = "allenai/MolmoAct2-SO100_101"

# ── Fine-tuned model repo ID (your HF username/model-name) ───────────────────
POLICY_REPO_ID  = "subhodipsaha/molmoact2_pick_place"

# ── Training mode ─────────────────────────────────────────────────────────────
TRAIN_MODE_VLM = "freeze"   # action expert only — VLM frozen

# ── Output ────────────────────────────────────────────────────────────────────
OUTPUT_DIR = "/content/drive/MyDrive/molmoact2_pick_place"

# ── Training hyperparams ──────────────────────────────────────────────────────
# Dataset: 17,960 frames
# Epochs = (BATCH_SIZE * STEPS) / 17960
#   bs=8,  steps=5000 → 40000 samples → 2.2 epochs
#   bs=16, steps=2500 → 40000 samples → 2.2 epochs  (same epochs, ~3h vs ~6h)
BATCH_SIZE  = 16
STEPS       = 2500
LOG_FREQ    = 100
SAVE_FREQ   = 500   # save more often since total steps are fewer

epochs = (BATCH_SIZE * STEPS) / 17960
print(f"Dataset    : {DATASET_REPO_ID}")
print(f"Checkpoint : {CHECKPOINT_PATH}")
print(f"Policy repo: {POLICY_REPO_ID}")
print(f"Train mode : {TRAIN_MODE_VLM}")
print(f"Batch size : {BATCH_SIZE}  |  Steps: {STEPS}  |  Epochs: {epochs:.1f}")
print(f"Output     : {OUTPUT_DIR}")

Dataset    : subhodipsaha/so101_pick_place_07_16
Checkpoint : allenai/MolmoAct2-SO100_101
Policy repo: subhodipsaha/molmoact2_pick_place
Train mode : freeze
Batch size : 16  |  Steps: 2500  |  Epochs: 2.2
Output     : /content/drive/MyDrive/molmoact2_pick_place


In [ ]:
import subprocess
import sys
import shutil
from pathlib import Path

# Delete leftover empty output dir so lerobot can create it fresh
ckpt_dirs = list(Path(OUTPUT_DIR).glob("checkpoints/*/pretrained_model")) if Path(OUTPUT_DIR).exists() else []
if Path(OUTPUT_DIR).exists() and not ckpt_dirs:
    shutil.rmtree(OUTPUT_DIR)
    print(f"Removed leftover output dir: {OUTPUT_DIR}")

cmd = [
    sys.executable, "-m", "lerobot.scripts.lerobot_train",
    f"--dataset.repo_id={DATASET_REPO_ID}",
    "--dataset.video_backend=pyav",
    "--policy.type=molmoact2",
    f"--policy.repo_id={POLICY_REPO_ID}",
    f"--policy.checkpoint_path={CHECKPOINT_PATH}",
    f"--policy.train_mode_vlm={TRAIN_MODE_VLM}",
    "--policy.action_mode=continuous",
    "--policy.model_dtype=bfloat16",
    "--policy.gradient_checkpointing=true",
    "--policy.normalize_gripper=true",  # gripper values in dataset are outside [-1,1]
    f"--batch_size={BATCH_SIZE}",
    f"--steps={STEPS}",
    f"--log_freq={LOG_FREQ}",
    f"--save_freq={SAVE_FREQ}",
    f"--output_dir={OUTPUT_DIR}",
    "--wandb.enable=false",
]

print("Running:", " ".join(cmd), flush=True)

process = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
for line in process.stdout:
    print(line, end="", flush=True)
process.wait()

if process.returncode != 0:
    raise RuntimeError(f"lerobot-train exited with code {process.returncode}")

Running: /usr/bin/python3 -m lerobot.scripts.lerobot_train --dataset.repo_id=subhodipsaha/so101_pick_place_07_16 --dataset.video_backend=pyav --policy.type=molmoact2 --policy.repo_id=subhodipsaha/molmoact2_pick_place --policy.checkpoint_path=allenai/MolmoAct2-SO100_101 --policy.train_mode_vlm=freeze --policy.action_mode=continuous --policy.model_dtype=bfloat16 --policy.gradient_checkpointing=true --policy.normalize_gripper=true --batch_size=16 --steps=2500 --log_freq=100 --save_freq=500 --output_dir=/content/drive/MyDrive/molmoact2_pick_place --wandb.enable=false
INFO 2026-09-13 03:13:13 ot_train.py:213 {'batch_size': 16,
 'checkpoint_path': None,
 'cudnn_deterministic': False,
 'dataset': {'episodes': None,
             'image_transforms': {'enable': False,
                                  'max_num_transforms': 3,
                                  'random_order': False,
                                  'tfs': {'affine': {'kwargs': {'degrees': [-5.0,
                                 

In [ ]:
print("hi")